# Phase C2 v2: Retrain with Pseudo-labels (Speed + Training Fixes)
Fixes from C2 v1:
- **Speed**: Pre-computed spectrograms + num_workers=16 (epoch: ~24min -> ~3-5min)
- **Training**: MixUp prob 0.5 (not 1.0), Beta(0.4,0.4) weights, drop_path=0.05 (not 0.15)
- **Validation**: Dual -- clean-clip AUC + soundscape AUC (model saved by soundscape AUC)

In [1]:
import os
import random
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

CFG = {
    'seed': 42,
    'base_dir': '/data/scratch/scienceteam/jupyter-mir/bird/Data/birdclef-2026',
    'spec_dir': '/data/scratch/scienceteam/jupyter-mir/bird/Data/precomputed_specs',
    'pseudo_path': '/data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed/pseudo_labels.csv',
    'output_dir': '/data/scratch/scienceteam/jupyter-mir/bird/models/stage2v2_sed',
    'sr': 32000,
    'duration': 20,
    'n_mels': 224,
    'img_width': 512,
    'model_name': 'tf_efficientnet_b0.ns_jft_in1k',
    'num_classes': 234,
    'batch_size': 180,
    'epochs': 30,
    'patience': 6,
    'lr': 5e-4,
    'weight_decay': 1e-4,
    'drop_path_rate': 0.05,
    'mixup_prob': 0.5,
    'mixup_alpha': 0.4,
    'n_folds': 5,
    'num_workers': 4,
}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])
os.makedirs(CFG['output_dir'], exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0)})")
print(f"Key changes from C2 v1: drop_path={CFG['drop_path_rate']}, mixup_prob={CFG['mixup_prob']}, "
      f"mixup_alpha={CFG['mixup_alpha']}, patience={CFG['patience']}, num_workers={CFG['num_workers']}")

Device: cuda (NVIDIA A40)
Key changes from C2 v1: drop_path=0.05, mixup_prob=0.5, mixup_alpha=0.4, patience=6, num_workers=4


In [2]:
from pathlib import Path

train = pd.read_csv(os.path.join(CFG['base_dir'], 'train.csv'))
taxonomy = pd.read_csv(os.path.join(CFG['base_dir'], 'taxonomy.csv'))

LABELS = sorted(taxonomy['primary_label'].tolist())
LABEL2IDX = {label: idx for idx, label in enumerate(LABELS)}
IDX2LABEL = {idx: label for label, idx in LABEL2IDX.items()}

# Build spec paths for clean clips (preserves species subfolder from filename)
train['spec_path'] = train['filename'].apply(
    lambda x: os.path.join(CFG['spec_dir'], 'train_audio', str(Path(x).with_suffix('.npy')))
)

# Pseudo-labels
pseudo_df = pd.read_csv(CFG['pseudo_path'])
pseudo_df['spec_path'] = pseudo_df.apply(
    lambda row: os.path.join(CFG['spec_dir'], 'soundscapes', f"{row['soundscape_id']}_{int(row['start_sec'])}.npy"),
    axis=1
)

# Labeled soundscapes (for validation only in C2 v2)
sc_labels = pd.read_csv(os.path.join(CFG['base_dir'], 'train_soundscapes_labels.csv'))

def parse_time(t):
    parts = t.split(':')
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

sc_labels['start_sec'] = sc_labels['start'].apply(parse_time)
sc_labels['soundscape_id'] = sc_labels['filename'].str.replace('.ogg', '', regex=False)
sc_labels['spec_path'] = sc_labels.apply(
    lambda row: os.path.join(CFG['spec_dir'], 'soundscapes', f"{row['soundscape_id']}_{int(row['start_sec'])}.npy"),
    axis=1
)

# Build label vectors for labeled soundscapes
sc_label_vectors = []
for _, row in sc_labels.iterrows():
    vec = np.zeros(CFG['num_classes'], dtype=np.float32)
    for species in str(row['primary_label']).split(';'):
        species = species.strip()
        if species in LABEL2IDX:
            vec[LABEL2IDX[species]] = 1.0
    sc_label_vectors.append(vec)
sc_labels['label_vector'] = sc_label_vectors

# Verify spec files exist
clean_exist = train['spec_path'].apply(os.path.exists).sum()
pseudo_exist = pseudo_df['spec_path'].apply(os.path.exists).sum()
sc_exist = sc_labels['spec_path'].apply(os.path.exists).sum()

print(f"Clean clips: {len(train):,} total, {clean_exist:,} specs found")
print(f"Pseudo-labeled: {len(pseudo_df):,} total, {pseudo_exist:,} specs found")
print(f"Labeled soundscapes (val only): {len(sc_labels):,} total, {sc_exist:,} specs found")
print(f"Target species: {len(LABELS)}")

assert clean_exist == len(train), f"Missing {len(train) - clean_exist} clean specs!"
assert pseudo_exist == len(pseudo_df), f"Missing {len(pseudo_df) - pseudo_exist} pseudo specs!"

Clean clips: 35,549 total, 35,549 specs found
Pseudo-labeled: 127,104 total, 127,104 specs found
Labeled soundscapes (val only): 1,478 total, 1,478 specs found
Target species: 234


In [3]:
class PrecomputedCleanDataset(Dataset):
    """Loads pre-computed spectrogram tensors for clean training clips."""

    def __init__(self, df, label2idx, cfg):
        self.spec_paths = df['spec_path'].values
        self.labels = df['primary_label'].values
        self.label2idx = label2idx
        self.num_classes = cfg['num_classes']

    def __len__(self):
        return len(self.spec_paths)

    def __getitem__(self, idx):
        arr = np.load(self.spec_paths[idx]).astype(np.float32)
        spec = torch.from_numpy(arr)
        target = torch.zeros(self.num_classes, dtype=torch.float32)
        label = self.labels[idx]
        if label in self.label2idx:
            target[self.label2idx[label]] = 1.0
        return spec, target


class PrecomputedSoundscapeDataset(Dataset):
    """Loads pre-computed spectrogram .npy files for soundscape segments (pseudo or labeled)."""

    def __init__(self, spec_paths, label_vectors):
        self.spec_paths = spec_paths
        self.label_vectors = label_vectors

    def __len__(self):
        return len(self.spec_paths)

    def __getitem__(self, idx):
        arr = np.load(self.spec_paths[idx]).astype(np.float32)
        spec = torch.from_numpy(arr)
        target = torch.tensor(self.label_vectors[idx], dtype=torch.float32)
        return spec, target


print("PrecomputedCleanDataset and PrecomputedSoundscapeDataset defined.")

PrecomputedCleanDataset and PrecomputedSoundscapeDataset defined.


In [4]:
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=-1).pow(1.0 / self.p)


class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg['model_name'], pretrained=True,
            in_chans=3, num_classes=0, global_pool='',
            drop_path_rate=cfg.get('drop_path_rate', 0.0),
        )
        self.feat_dim = 1280
        self.freq_pool = GeM(p=3)
        self.fc = nn.Linear(self.feat_dim, cfg['num_classes'])

    def forward(self, x):
        features = self.backbone(x)
        x = self.freq_pool(features)
        x = x.transpose(1, 2)
        frame_logits = self.fc(x)
        clip_logits = frame_logits.mean(dim=1)
        clip_logits_max = frame_logits.max(dim=1)[0]
        output = 0.5 * clip_logits + 0.5 * clip_logits_max
        return output


model = SEDModel(CFG).to(DEVICE)
dummy = torch.randn(2, 3, 224, 512).to(DEVICE)
out = model(dummy)
print(f"Output shape: {out.shape}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Stochastic Depth: drop_path_rate={CFG['drop_path_rate']}")
del model, dummy, out
torch.cuda.empty_cache()

Output shape: torch.Size([2, 234])
Total parameters: 4,307,303
Stochastic Depth: drop_path_rate=0.05


In [5]:
def mixup(mel, target, mel2, target2, alpha=0.4):
    """MixUp with Beta-distributed blend weight. Target: element-wise max."""
    lam = np.random.beta(alpha, alpha)
    mel_mixed = lam * mel + (1 - lam) * mel2
    target_mixed = torch.max(target, target2)
    return mel_mixed, target_mixed


def spec_augment(mel, freq_mask=30, time_mask=50):
    _, n_mels, n_time = mel.shape
    f = random.randint(0, freq_mask)
    f0 = random.randint(0, max(0, n_mels - f))
    mel[:, f0:f0+f, :] = 0
    t = random.randint(0, time_mask)
    t0 = random.randint(0, max(0, n_time - t))
    mel[:, :, t0:t0+t] = 0
    return mel


print(f"MixUp (prob={CFG['mixup_prob']}, alpha={CFG['mixup_alpha']}) and SpecAugment defined.")

MixUp (prob=0.5, alpha=0.4) and SpecAugment defined.


In [6]:
scaler = torch.amp.GradScaler()

def train_one_epoch_c2v2(model, clean_loader, pseudo_loader, optimizer, scheduler, device, cfg):
    model.train()
    losses = []
    criterion = nn.CrossEntropyLoss()
    pseudo_iter = iter(pseudo_loader)
    mix_count = 0
    pure_count = 0

    pbar = tqdm(clean_loader, desc="Train")
    for mel_clean, target_clean in pbar:
        mel_clean = mel_clean.to(device)
        target_clean = target_clean.to(device)

        use_mixup = random.random() < cfg['mixup_prob']

        if use_mixup:
            try:
                mel_pseudo, target_pseudo = next(pseudo_iter)
            except StopIteration:
                pseudo_iter = iter(pseudo_loader)
                mel_pseudo, target_pseudo = next(pseudo_iter)

            mel_pseudo = mel_pseudo.to(device)
            target_pseudo = target_pseudo.to(device)

            bs = min(mel_clean.size(0), mel_pseudo.size(0))
            mel_clean, target_clean = mel_clean[:bs], target_clean[:bs]
            mel_pseudo, target_pseudo = mel_pseudo[:bs], target_pseudo[:bs]

            mel, target = mixup(mel_clean, target_clean, mel_pseudo, target_pseudo,
                                alpha=cfg['mixup_alpha'])
            mix_count += 1
        else:
            mel, target = mel_clean, target_clean
            pure_count += 1

        for i in range(mel.size(0)):
            if random.random() < 0.5:
                mel[i] = spec_augment(mel[i])

        with torch.amp.autocast(device_type='cuda'):
            logits = model(mel)
            loss = criterion(logits, target)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(loss.item())
        pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.4f}")

    scheduler.step()
    return np.mean(losses), mix_count, pure_count


print("train_one_epoch_c2v2 defined (with AMP mixed precision).")

train_one_epoch_c2v2 defined (with AMP mixed precision).


In [7]:
def validate(model, loader, device, desc="Valid"):
    model.eval()
    all_preds = []
    all_targets = []
    losses = []
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad(), torch.amp.autocast(device_type='cuda'):
        for mel, target in tqdm(loader, desc=desc):
            mel, target = mel.to(device), target.to(device)
            logits = model(mel)
            loss = criterion(logits, target)
            losses.append(loss.item())
            preds = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(target.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    aucs = []
    for i in range(all_targets.shape[1]):
        if all_targets[:, i].sum() > 0:
            try:
                auc = roc_auc_score(all_targets[:, i], all_preds[:, i])
                aucs.append(auc)
            except ValueError:
                pass

    macro_auc = np.mean(aucs) if aucs else 0.0
    return np.mean(losses), macro_auc, len(aucs)


print("validate defined (used for both clean and soundscape validation).")

validate defined (used for both clean and soundscape validation).


In [ ]:
# Build pseudo-label arrays
pseudo_label_vectors = pseudo_df[LABELS].values.astype(np.float32)
pseudo_spec_paths = pseudo_df['spec_path'].values

# Soundscape validation set (all 1,478 labeled segments -- never used for training)
sc_val_spec_paths = sc_labels['spec_path'].values
sc_val_label_vecs = np.stack(sc_labels['label_vector'].values).astype(np.float32)

sc_val_ds = PrecomputedSoundscapeDataset(sc_val_spec_paths, sc_val_label_vecs)
sc_val_loader = DataLoader(sc_val_ds, batch_size=CFG['batch_size'], shuffle=False,
                           num_workers=CFG['num_workers'], pin_memory=True)

print(f"Pseudo-labeled pool: {len(pseudo_df):,} segments")
print(f"Soundscape validation set: {len(sc_val_ds):,} segments (never in training)")

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train['primary_label'])):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    train_df = train.iloc[train_idx]
    val_df = train.iloc[val_idx]
    print(f"Clean train: {len(train_df):,} | Clean val: {len(val_df):,}")

    # Clean data loaders
    train_ds = PrecomputedCleanDataset(train_df, LABEL2IDX, CFG)
    val_ds = PrecomputedCleanDataset(val_df, LABEL2IDX, CFG)

    clean_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                              num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    clean_val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                                  num_workers=CFG['num_workers'], pin_memory=True)

    # Pseudo-label loader (shuffled, cycles via train loop)
    pseudo_ds = PrecomputedSoundscapeDataset(pseudo_spec_paths, pseudo_label_vectors)
    pseudo_loader = DataLoader(pseudo_ds, batch_size=CFG['batch_size'], shuffle=True,
                               num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)

    print(f"Pseudo pool: {len(pseudo_ds):,} segments")

    # Model
    model = SEDModel(CFG).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, eta_min=1e-6)

    best_sc_auc = 0
    best_clean_auc = 0
    no_improve = 0

    for epoch in range(1, CFG['epochs'] + 1):
        print(f"\n--- Epoch {epoch}/{CFG['epochs']} ---")

        train_loss, n_mix, n_pure = train_one_epoch_c2v2(
            model, clean_loader, pseudo_loader, optimizer, scheduler, DEVICE, CFG
        )

        # Dual validation
        clean_val_loss, clean_val_auc, n_clean_scored = validate(model, clean_val_loader, DEVICE, desc="CleanVal")
        sc_val_loss, sc_val_auc, n_sc_scored = validate(model, sc_val_loader, DEVICE, desc="SC_Val")

        print(f"Train loss: {train_loss:.4f} (mix={n_mix}, pure={n_pure})")
        print(f"Clean val -- loss: {clean_val_loss:.4f} | AUC: {clean_val_auc:.4f} ({n_clean_scored} species)")
        print(f"Sound val -- loss: {sc_val_loss:.4f} | AUC: {sc_val_auc:.4f} ({n_sc_scored} species)")

        if sc_val_auc > best_sc_auc:
            best_sc_auc = sc_val_auc
            best_clean_auc = clean_val_auc
            no_improve = 0
            save_path = os.path.join(CFG['output_dir'], f'c2v2_fold{fold}.pth')
            torch.save(model.state_dict(), save_path)
            print(f"  -> Saved best model (SC_AUC={best_sc_auc:.4f}, Clean_AUC={best_clean_auc:.4f})")
        else:
            no_improve += 1
            print(f"  -> No improvement ({no_improve}/{CFG['patience']})")
            if no_improve >= CFG['patience']:
                print(f"  -> Early stopping at epoch {epoch}")
                break

    fold_results.append({
        'fold': fold,
        'best_sc_auc': best_sc_auc,
        'best_clean_auc': best_clean_auc,
    })
    print(f"\nFold {fold} best: SC_AUC={best_sc_auc:.4f}, Clean_AUC={best_clean_auc:.4f}")

    del model, optimizer, scheduler, clean_loader, clean_val_loader, pseudo_loader
    del train_ds, val_ds, pseudo_ds
    torch.cuda.empty_cache()

Pseudo-labeled pool: 127,104 segments
Soundscape validation set: 1,478 segments (never in training)

FOLD 0
Clean train: 28,439 | Clean val: 7,110
Pseudo pool: 127,104 segments

--- Epoch 1/30 ---


SC_Val: 100%|██████████| 9/9 [00:05<00:00,  1.77it/s]


Train loss: 254.1556 (mix=81, pure=76)
Clean val -- loss: 5.0703 | AUC: 0.7498 (198 species)
Sound val -- loss: 26.9559 | AUC: 0.7124 (75 species)
  -> Saved best model (SC_AUC=0.7124, Clean_AUC=0.7498)

--- Epoch 2/30 ---


SC_Val: 100%|██████████| 9/9 [00:01<00:00,  4.91it/s]

Train loss: 224.2692 (mix=72, pure=85)
Clean val -- loss: 4.8403 | AUC: 0.8734 (198 species)
Sound val -- loss: 26.4717 | AUC: 0.6550 (75 species)
  -> No improvement (1/6)

--- Epoch 3/30 ---



SC_Val: 100%|██████████| 9/9 [00:02<00:00,  4.12it/s]


Train loss: 211.7343 (mix=68, pure=89)
Clean val -- loss: 4.8384 | AUC: 0.8876 (198 species)
Sound val -- loss: 26.6092 | AUC: 0.6487 (75 species)
  -> No improvement (2/6)

--- Epoch 4/30 ---


Train:  17%|█▋        | 27/157 [00:19<01:23,  1.55it/s, loss=270.4715]

In [ ]:
results_df = pd.DataFrame(fold_results)
print("\n" + "="*60)
print("C2 v2 RESULTS SUMMARY")
print("="*60)
print(results_df.to_string(index=False))
print(f"\nMean SC_AUC:    {results_df['best_sc_auc'].mean():.4f} +/- {results_df['best_sc_auc'].std():.4f}")
print(f"Mean Clean_AUC: {results_df['best_clean_auc'].mean():.4f} +/- {results_df['best_clean_auc'].std():.4f}")
print(f"\nModels saved to: {CFG['output_dir']}")

print("\n--- Comparison ---")
print(f"C1 clean-val AUC:    0.9773 (baseline)")
print(f"C2 v1 clean-val AUC: ~0.84 (over-regularized)")
print(f"C2 v2 clean-val AUC: {results_df['best_clean_auc'].mean():.4f}")
print(f"C2 v2 SC-val AUC:    {results_df['best_sc_auc'].mean():.4f} (new metric!)")

In [ ]:
!df -h /dev/shm

In [ ]:
!sudo umount /dev/shm && sudo mount -t tmpfs -o rw,size=64G tmpfs /dev/shm

In [ ]:
!sudo umount /dev/shm && !sudo mount -t tmpfs -o rw,size=64G tmpfs /dev/shm

In [ ]:
!rm -rf /data/scratch/scienceteam/jupyter-mir/bird/Data/precomputed_specs/train_audio/